## Cài đặt thư viện

In [45]:
!pip install underthesea -q

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from underthesea import word_tokenize
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns
from tqdm import tqdm

## Tải corpus tiếng Việt (Ted talks)

In [46]:
print("Đang tải corpus tiếng Việt TED talks...")
!wget https://raw.githubusercontent.com/thandongtb/convert_tieq_viet/master/data/Vietnamese.txt -O vi_ted_corpus.txt

with open("vi_ted_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Word segmentation tốt (underthesea)
tokens = []
lines = text.split("\n")
for line in tqdm(lines[:50000]):          # ~500k tokens
    if line.strip():
        tokens.extend(word_tokenize(line.lower()))

print(f"Số tokens: {len(tokens)}")
print("Ví dụ 15 từ đầu:", tokens[:15])

Đang tải corpus tiếng Việt TED talks...
--2026-03-16 16:40:55--  https://raw.githubusercontent.com/thandongtb/convert_tieq_viet/master/data/Vietnamese.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18332188 (17M) [text/plain]
Saving to: ‘vi_ted_corpus.txt’

vi_ted_corpus.txt   100%[===================>]  17.48M  53.2MB/s    in 0.3s    

2026-03-16 16:40:55 (53.2 MB/s) - ‘vi_ted_corpus.txt’ saved [18332188/18332188]



100%|██████████| 50000/50000 [00:38<00:00, 1293.54it/s]

Số tokens: 389902
Ví dụ 15 từ đầu: ['đối với', 'tôi', ',', 'câu chuyện', 'này', 'bắt đầu', '15', 'năm', 'trước', ',', 'khi', 'tôi', 'còn', 'là', 'một']


## Tạo vocab + corpus

In [47]:
min_count = 4
counter = Counter(tokens)
vocab = [word for word, freq in counter.items() if freq >= min_count]
word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for w, i in word2id.items()}

corpus = [word2id[w] for w in tokens if w in word2id]

corpus = corpus[:400000]
print(f"Vocab size: {len(vocab)}")

Vocab size: 4742


## Tạo positive pairs + negative distribution

In [48]:
def generate_pairs(corpus, window=2):
    pairs = []
    for i, target in enumerate(corpus):
        for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
            if i != j:
                pairs.append((target, corpus[j]))
    return pairs

pairs = generate_pairs(corpus)
print(f"Số positive pairs: {len(pairs)}")

word_freq = np.array([counter[id2word[i]] for i in range(len(vocab))], dtype=np.float64)
powered = word_freq ** 0.75
neg_dist = powered / powered.sum()

print(f"Sum của neg_dist: {neg_dist.sum():.6f}")

Số positive pairs: 1506302
Sum của neg_dist: 1.000000


## Dataset và Model SGNS

In [49]:
class SGNS_Dataset(Dataset):
    def __init__(self, pairs, neg_dist, num_neg=5):
        self.pairs = pairs
        self.neg_dist = neg_dist
        self.num_neg = num_neg
        self.vocab_size = len(neg_dist)

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        target, context = self.pairs[idx]
        negatives = np.random.choice(self.vocab_size, self.num_neg, p=self.neg_dist)
        return torch.tensor(target), torch.tensor(context), torch.tensor(negatives)

class SGNS(nn.Module):
    def __init__(self, vocab_size, dim=100):
        super().__init__()
        self.target_emb = nn.Embedding(vocab_size, dim)
        self.context_emb = nn.Embedding(vocab_size, dim)
        self.init_weights()

    def init_weights(self):
        init = 0.5 / 100
        self.target_emb.weight.data.uniform_(-init, init)
        self.context_emb.weight.data.uniform_(-init, init)

    def forward(self, target, pos_context, neg_context):
        t_emb = self.target_emb(target)                    # [B, D]
        pos_emb = self.context_emb(pos_context)            # [B, D]
        neg_emb = self.context_emb(neg_context)            # [B, K, D]

        pos_score = torch.sum(t_emb * pos_emb, dim=1)
        neg_score = torch.bmm(neg_emb, t_emb.unsqueeze(2)).squeeze(2)

        loss = -F.logsigmoid(pos_score).mean() - F.logsigmoid(-neg_score).mean()
        return loss

    def get_emb(self):
        return (self.target_emb.weight + self.context_emb.weight).detach().cpu().numpy()

## Train model

In [50]:
dataset = SGNS_Dataset(pairs, neg_dist)
loader = DataLoader(dataset, batch_size=1024, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SGNS(len(vocab), dim=100).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0025)

for epoch in range(4):
    total_loss = 0
    for t, p, n in tqdm(loader):
        t, p, n = t.to(device), p.to(device), n.to(device)
        optimizer.zero_grad()
        loss = model(t, p, n)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} - Loss: {total_loss/len(loader):.4f}")

100%|██████████| 1471/1471 [05:06<00:00,  4.79it/s]


Epoch 1 - Loss: 1.2512


100%|██████████| 1471/1471 [05:02<00:00,  4.87it/s]


Epoch 2 - Loss: 1.1711


100%|██████████| 1471/1471 [04:57<00:00,  4.94it/s]


Epoch 3 - Loss: 1.1300


100%|██████████| 1471/1471 [04:55<00:00,  4.98it/s]

Epoch 4 - Loss: 1.1013


## Lấy embeddings và các hàm hỗ trợ

In [51]:
embeddings = model.get_emb()

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def most_similar(word, topn=8):
    if word not in word2id: return []
    idx = word2id[word]
    vec = embeddings[idx]
    sims = [(id2word[i], cosine_sim(vec, embeddings[i]))
            for i in range(len(embeddings)) if i != idx]
    return sorted(sims, key=lambda x: x[1], reverse=True)[:topn]

## Kiểm tra một số ví dụ

In [52]:
print("=== ĐÁNH GIÁ EMBEDDINGS ===")

test_pairs = [
    ("học", "máy"),
    ("trí", "tuệ"),
    ("dữ", "liệu"),
    ("bệnh", "viện"),
    ("bác", "sĩ"),
    ("xe", "ô"),
    ("âm", "nhạc"),
    ("bài", "hát"),
    ("học", "máy"),
    ("trí", "tuệ"),
    ("thông", "minh"),
    ("khoa", "học")
]

for a, b in test_pairs:
    if a in word2id and b in word2id:
        score = cosine_sim(embeddings[word2id[a]], embeddings[word2id[b]])
        print(f"{a} ↔ {b}: {score:.4f}")
    else:
        print(f"{a} ↔ {b}: (một trong hai từ không có trong vocab)")

=== ĐÁNH GIÁ EMBEDDINGS ===
học ↔ máy: 0.0224
trí ↔ tuệ: (một trong hai từ không có trong vocab)
dữ ↔ liệu: (một trong hai từ không có trong vocab)
bệnh ↔ viện: -0.1088
bác ↔ sĩ: (một trong hai từ không có trong vocab)
xe ↔ ô: 0.2849
âm ↔ nhạc: 0.2054
bài ↔ hát: 0.6909
học ↔ máy: 0.0224
trí ↔ tuệ: (một trong hai từ không có trong vocab)
thông ↔ minh: (một trong hai từ không có trong vocab)
khoa ↔ học: 0.1628
